In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Modules and Model Construction

The multilayer perceptron of that section had four
parameter arrays, and we could name each one by hand. The models in the rest
of this book have many more. A residual network for images stacks dozens of
layers, and the language model Llama 3 8B
[@Grattafiori.Dubey.Jauhri.ea.2024], with 8 billion parameters,
consists of 32 identical blocks and stores its parameters in 291 arrays.
Every one of them must
reach the optimizer, be written to a checkpoint, and move to the GPU. An
array that is left out raises no error: it is simply never trained, or never
saved.

Deep learning frameworks solve this with one abstraction, the *module*. A
module is an object that owns parameters, owns other modules (its
*children*), and defines a forward computation from inputs to outputs.
Layers, groups of layers, and whole models are all modules, so a model is a
tree of modules, as in the figure. The framework finds every
parameter by walking this tree, provided that each child has been
*registered* with its parent. This section shows how registration works, how
it fails, and how to write the forward computation of a module yourself. The
rest of the chapter uses the same walk for other tasks: counting the
parameters (that section), initializing them
(that section), saving them (that section), and
moving them to a GPU (that section).

![The module tree of the residual MLP that we build at the end of this section. Each edge carries the name under which the parent registered the child, and the path of names from the root to a parameter is the name of that parameter, here `1.body.0.weight`. Blocks 2 to 4 repeat block 1. PyTorch and Gluon join the names with dots; JAX lists the same path as a tuple and keeps the children of a sequential container under `layers`. Gluon folds the ReLU into the first dense layer, so its second dense layer is child 1 of the body; Keras names the children after their classes.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-names.svg)

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
from dataclasses import dataclass

## The Module Abstraction

### Sequential Containers and the Registry

Every model since that section has been a module: `d2l.Module`
is a subclass of the framework's module class. The quickest way to build a
module from layers is a *sequential container*, a module that applies its
children one after the other. Here it holds the MLP of
that section, with a hidden layer of 256 ReLU units and
10 outputs, applied to a minibatch of two examples with 20 features each.

In PyTorch the module class is `nn.Module`, and the container
`nn.Sequential` takes its layers as arguments.

In [2]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))

X = torch.rand(2, 20)
net(X).shape

torch.Size([2, 10])

The container keeps its children in a *registry*, a table from names to
modules, and its forward computation runs through that table in order.

`named_children` lists the registry. The names are the positions `'0'`,
`'1'`, and `'2'`; the ReLU is a module without parameters.

In [3]:
dict(net.named_children())

{'0': Linear(in_features=20, out_features=256, bias=True),
 '1': ReLU(),
 '2': Linear(in_features=256, out_features=10, bias=True)}

Everything the framework does to a model as a whole walks this registry.
Collecting the parameters for the optimizer, writing them to a file, and
moving them to a GPU all visit the registered children, then their children,
and so on down to the parameters. A layer that is not registered is skipped
by all of them.

### Subclasses of the Module Class

A sequential container only expresses a chain. For any other wiring we
subclass the module class and write two methods: a constructor that creates
the children, and a forward method that applies them to the input.

In PyTorch the forward method is `forward`.

Here is the same MLP once more:

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [5]:
net = MLP()
net(X).shape

torch.Size([2, 10])

Two details make this work. First, `self.hidden = ...` is not an ordinary
attribute assignment. The module class intercepts assignment (the Python
method `__setattr__`), and when the value is a module it enters the
registry under the attribute name, here `hidden` and `out`. That is all it
takes for both layers' parameters to reach the optimizer. Second, we wrote
no backward method: automatic differentiation derives the gradients from
whatever the forward method computes.

We call the model as `net(X)`, never as `net.forward(X)`. Calling a module
runs `nn.Module.__call__`, which runs `forward` and, around it, any *hooks*
registered on the module: functions that see its inputs and outputs without
changing its code. that section uses them to inspect a model.

## Module Containers

### Writing a Container

A container has two jobs: register each child, and run the children in
order. Writing one ourselves shows both. The registration step is the one
that can go wrong.

In [6]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self.add_module(str(idx), module)

    def forward(self, X):
        for module in self.children():
            X = module(X)
        return X

`add_module` enters a child into the registry under a name, here its
position as a string, and `children()` runs through the registry in the
order of insertion.

Our container works like the built-in one:

In [7]:
net = MySequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
net(X).shape

torch.Size([2, 10])

### Layers in a Plain List

Now suppose we store the layers in an ordinary Python list. The forward
method can still loop over the list, but the list itself is not a module.
Does the framework still find the layers inside it?

The module below keeps its layers in a plain list:

In [8]:
class PlainListMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10)]

    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

net = PlainListMLP()
net(X).shape, sum(p.numel() for p in net.parameters())

(torch.Size([2, 10]), 0)

The model computes, yet it owns zero parameters. A plain Python list is not
a module, so `__setattr__` stores it like any other value and registers
nothing inside it. The layers still hold their own weights, which is why
the forward pass runs. But `net.parameters()` is empty, `net.to(device)`
leaves every layer on the CPU, and `net.state_dict()` saves an empty model.
Here the optimizer at least complains: `torch.optim.SGD` refuses an empty
list of parameters. A model that is only partly registered is worse.
Suppose its input and output layers are attributes and the blocks between
them sit in a plain list. On the CPU, the optimizer then trains the two
outer layers and leaves the blocks at their initial weights, without any
error, and the only sign is a model that learns less than it should. On a
GPU the failure is at least loud: `Trainer(num_gpus=1)` moves only the
registered layers, and the first forward pass stops with an error about
tensors on different devices. The registered
container for a list of children is `nn.ModuleList`:

In [9]:
class ModuleListMLP(PlainListMLP):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList(self.layers)

net = ModuleListMLP()
net(X).shape, sum(p.numel() for p in net.parameters())

(torch.Size([2, 10]), 7946)

Same forward pass, and now all 7,946 parameters are registered: 20 · 256 +
256 in the hidden layer and 256 · 10 + 10 in the output layer. The
containers divide the work as follows. `nn.Sequential` registers its
children and supplies a forward method that chains them. `nn.ModuleList`
registers a list of children and supplies no forward method, so we write
the loop ourselves, which is what we want when the loop is not a plain
chain: blocks that take an extra argument, or a computation between blocks.
`nn.ModuleDict` does the same for children indexed by name. A model that
repeats one block many times, such as the 32-block 8B model of the
introduction, keeps the blocks in an `nn.ModuleList`.

## Forward Computation

The forward method is ordinary Python. It can call any tensor operation,
loop, branch, and combine intermediate results. This section uses that
freedom three times: for a residual connection, for a constant, and for a
loop whose length depends on the data.

### Residual Connections

The most important example in this book is the *residual connection*,
which adds the input of a block to its output. If $f$ is the *body* of the
block, the computation inside it, here two fully connected layers with a
ReLU between them, the block computes

$$\mathbf{y} = \mathbf{x} + f(\mathbf{x}).$$

The block outputs its input plus a correction $f(\mathbf{x})$. If $f$ is
zero, the block is the identity, so a stack of such blocks can pass its
input through unchanged, and each block only has to learn a correction to
its input. The networks of that section are built this way. No
framework provides this as a layer; it is one line of the forward method,
`X + self.body(X)`:

In [10]:
class ResidualBlock(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(num_hiddens, num_hiddens), nn.ReLU(),
            nn.Linear(num_hiddens, num_hiddens))

    def forward(self, X):
        return X + self.body(X)

![A residual block. The input splits into the body, Linear, ReLU, Linear, and an identity path, and the two rejoin by addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

the figure shows the wiring. The skip path also
changes the backward pass. The Jacobian of the block is
$\mathbf{I} + \mathbf{J}_f$, where $\mathbf{J}_f$ is the Jacobian of the
body, so the gradient that reaches $\mathbf{x}$ is the gradient at
$\mathbf{y}$ plus the part that flows through the body
[@He.Zhang.Ren.ea.2016*1]. The addition also has a consequence for
construction: $\mathbf{x}$ and $f(\mathbf{x})$ must have the same shape, so
a residual block has a single width, which we pass to the constructor.

The body therefore uses `nn.Linear` with both widths rather than lazy
layers.

In [11]:
block = ResidualBlock(24)
block(torch.randn(2, 24)).shape

torch.Size([2, 24])

### Constants in the Forward Method

The forward method may also use values that are neither inputs nor
parameters. Suppose we want to damp the correction by a fixed factor
$\alpha = 0.5$, so that the block computes
$\mathbf{x} + \alpha f(\mathbf{x})$. The factor is part of the design, and
training must not change it:

In [12]:
class ScaledResidual(ResidualBlock):
    def __init__(self, num_hiddens, alpha=0.5):
        super().__init__(num_hiddens)
        self.alpha = torch.tensor(alpha)  # Fixed by design, never trained

    def forward(self, X):
        return X + self.alpha * self.body(X)

block = ScaledResidual(24)
'alpha' in block.state_dict(), list(block.state_dict())[:2]

(False, ['body.0.weight', 'body.0.bias'])

The factor enters the computation but is not a parameter: it is missing
from `named_parameters()`, so the optimizer never changes it, which is what
we wanted. It is missing from `state_dict()` as well, so it is not saved
with the model, and `.to(device)` does not move it. Some state must travel
with the model without being trained. that section gives such
state its registered place, a *buffer*.

### Eager and Compiled Execution

A forward method is ordinary Python code, and so far it has also run like
ordinary Python, one line at a time. A framework can instead record the
computation once and replay the recording, which is faster but restricts
what the forward method may do.

PyTorch runs `forward` as written, one operation at a time, a mode called
*eager execution*, and so does our `Trainer`. A loop whose condition depends
on the values of tensors therefore works just as it does in NumPy.
Compilers for PyTorch models exist, and that section uses
them.

## Lazy Initialization

### Shapes from Data

Since that section, our PyTorch models have used lazy
layers such as `nn.LazyLinear(256)`, which name only their output width.
The weight has shape `(256, in_features)`, and we never said what
`in_features` is: the layer reads it from the first minibatch, and the
`Trainer` initializes the weights afterwards by calling the model's method
`weight_init`. Here is the mechanism. Before the first
forward pass the weight is a placeholder without a shape:

In [13]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
net[0].weight

<UninitializedParameter>

The first forward pass reads the input width from the minibatch, creates
the weight, and turns the layer into an ordinary `nn.Linear`. The output
width of each layer is the input width of the next, so the shapes cascade
through the model:

In [14]:
net(X)
net[0].weight.shape

torch.Size([256, 20])

### Initialization after a Dry Run

Lazy layers spare us the arithmetic of input widths: in a deep model the
input width of each layer is the output width of the one before, and
working it out by hand is tedious and easy to get wrong. The cost is that
the parameters do not exist until a minibatch has passed through. Before
that, they cannot be counted, and an initializer whose scale depends on the
number of inputs, such as the Xavier initialization of
that section, cannot be applied. Initialization therefore follows
a *dry run*, one forward pass on a representative minibatch whose output is
discarded.

`d2l.Module` packs the dry run and the initialization into one method,
`apply_init`. It runs the model once without recording gradients, then
calls the given function on every submodule. The `Trainer` calls
`apply_init` with the model's `weight_init` before training whenever the
model still has lazy parameters, as in the training runs of the previous
chapters.

In [15]:
@d2l.add_to_class(d2l.Module)
def apply_init(self, inputs, init=None):
    """Run a dry pass to fix lazy shapes, then apply `init`."""
    with torch.no_grad():
        self(*inputs)
    if init is not None:
        self.apply(init)

`nn.Module.apply(fn)` calls `fn` on every module of the tree, children
before their parents: one more task that is a walk over the tree. Here is
`apply_init` with Xavier's uniform initializer:

In [16]:
class TinyMLP(d2l.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(),
                                 nn.LazyLinear(10))

def init_xavier(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)

model = TinyMLP()
model.apply_init([X], init_xavier)
model.net[0].weight.shape

torch.Size([256, 20])

The dry run turned every lazy layer into an `nn.Linear`, after which
`init_xavier` could recognize the layers and redraw their weights.

## Models from a Config

### Configs and Builders

A trained model is a set of arrays plus the code that knows what to do with
them. The arrays alone do not say how to rebuild the model. Every width and
depth in this section so far was a literal typed into a constructor call,
and a checkpoint of the weights records such choices at best indirectly:
the shapes of the arrays reveal the widths, but nothing in them reveals a
choice without parameters, such as the activation function. The choices that
vary from one experiment to the next therefore belong in a small
*configuration* object, or *config*, that is logged with the results and
saved with the weights. The topology, meaning what a block computes and how
blocks connect, stays in code. Here a Python `dataclass` holds four fields,
and `build` turns them into a model:

In [17]:
@dataclass
class MLPConfig:
    num_inputs: int = 784
    num_hiddens: int = 256
    num_blocks: int = 4
    num_outputs: int = 10

def build(cfg: MLPConfig) -> nn.Module:
    blocks = [ResidualBlock(cfg.num_hiddens) for _ in range(cfg.num_blocks)]
    return nn.Sequential(nn.Linear(cfg.num_inputs, cfg.num_hiddens),
                         *blocks, nn.Linear(cfg.num_hiddens, cfg.num_outputs))

One config gives one architecture: an input layer from `num_inputs` features to
the hidden width, `num_blocks` residual blocks, and an output layer. The
blocks pass through a plain Python list, which is safe here because
unpacking it into the sequential container registers each block. Every
layer receives both of its widths, so the model needs no dry run before its
parameters can be counted.

The names of the first parameters are the paths of the figure,
joined by dots:

In [18]:
net = build(MLPConfig())
[(name, tuple(p.shape)) for name, p in net.named_parameters()][:4]

[('0.weight', (256, 784)),
 ('0.bias', (256,)),
 ('1.body.0.weight', (256, 256)),
 ('1.body.0.bias', (256,))]

The model maps a minibatch of 784-dimensional inputs, the size of a
flattened Fashion-MNIST image, to 10 outputs:

In [19]:
net(torch.rand(2, 784)).shape

torch.Size([2, 10])

### Counting Parameters

The config also tells us the size of the model before we build it. The
input layer has $784 \cdot 256 + 256 = 200{,}960$ parameters, each block
has two layers of $256 \cdot 256 + 256$, or $131{,}584$ together, and the
output layer has $256 \cdot 10 + 10 = 2{,}570$. The default model therefore
has

$$200{,}960 + 4 \cdot 131{,}584 + 2{,}570 = 729{,}866$$

parameters, most of them in the blocks. Rescaling the model is a change to
two fields, not an edit of the model code, and a walk over the parameters
confirms the count:

In [20]:
for cfg in (MLPConfig(), MLPConfig(num_hiddens=512, num_blocks=8)):
    n = sum(p.numel() for p in build(cfg).parameters())
    print(f'num_hiddens={cfg.num_hiddens}, num_blocks={cfg.num_blocks}: '
          f'{n:,} parameters')

num_hiddens=256, num_blocks=4: 729,866 parameters
num_hiddens=512, num_blocks=8: 4,609,546 parameters


Doubling the width multiplies the parameters of each block by about four,
and doubling the number of blocks doubles their total, so the larger
config has more than six times as many parameters. The 8B model of the
introduction is described the same way: a config of
a dozen numbers, such as 32 blocks and a width of 4,096, and code that
turns it into a module tree.

A config records numbers, not code. Rebuilding a model from a checkpoint
also needs the same `build` function and the same block classes, which is
why that section saves the config inside the checkpoint next to
the weights.


We can now build a model of any size from modules, find all of its
parameters, and rebuild it from a few numbers. What we have not yet done is
look at the parameters themselves: which tensors a model holds, how much
memory they take, and how to share or freeze them. That is the subject of
that section.

## Summary

A module owns parameters, child modules, and a forward computation. Layers,
blocks, and whole models are modules, so a model is a tree, and the name of
each parameter is its path from the root. The framework finds parameters by
walking the registered children, so a child stored where the walk cannot
see it is never trained or saved. The forward method is ordinary Python, so
a residual connection is one line of it, although a compiled training step
forbids control flow that depends on the values of arrays. Lazy layers take
their input widths from the first minibatch, so their initialization follows
a dry run. A config of a few numbers, together with a `build` function,
records an architecture so that it can be logged, saved, and rebuilt.

In PyTorch, children are registered by attribute assignment and by the
containers `nn.Sequential`, `nn.ModuleList`, and `nn.ModuleDict`. Modules in
a plain list stay unregistered, without any warning.

## Exercises

1. [code] **Unregistered modules.** The plain list in `PlainListMLP` keeps
   its layers out of `net.parameters()`, `net.to(device)`, and
   `net.state_dict()`. Predict, then verify, what three further operations
   do to the model.
    1. Call `net.apply(init_xavier)` and check whether the weight of
       `net.layers[0]` changes.
    1. Add `nn.Dropout(0.5)` after the `nn.ReLU()` in the list, call
       `net.eval()`, and pass the same input through the model twice.
    1. The method `load_state_dict` is the inverse of `state_dict`: it
       copies a dictionary of saved tensors into a model. Call
       `net.load_state_dict(ModuleListMLP().state_dict())`.
    1. Which of these failures are silent and which raise an error? Explain
       all three from the same missing registration.

2. **Widening a residual block.** `ResidualBlock` requires its input and
   output widths to agree. Suppose a block maps width $d$ to a larger width
   $d'$, so its body ends in a $d'$-wide layer and the skip path must also
   produce width $d'$. Give one way to build that skip path with
   parameters and one without. For each, count the extra parameters and
   state which output coordinates still receive the identity path.

3. [code] **Config boundaries.** Extend `MLPConfig` with a field
   `act: str = 'relu'` that selects the activation inside every
   `ResidualBlock`, and make `build` honor it.
    1. Build one model with `act='tanh'` and one from the default config,
       and compare their parameter shapes. Could a checkpoint that holds
       only the weights tell the two models apart? State what the answer
       implies for the config.
    1. A plain feed-forward block has the same `body` as `ResidualBlock`
       but no skip connection. Apply the same test to the choice between
       the two block types, and decide whether that choice belongs in the
       config or in code.

4. [code] **A module from scratch.** ● In plain Python and NumPy, without any
   framework base class, write a `Parameter` class that wraps an array and
   a minimal `Module` class whose `__setattr__` registers every assigned
   `Parameter` or `Module`. Give `Module` a `parameters()` method that
   recurses into the registered children. Write a `Linear` subclass of
   `Module` that holds a weight and a bias, build a two-layer MLP from two
   instances of it, and verify that `parameters()` returns all four arrays.
   Then store the two layers in a plain list, check what `parameters()`
   returns, and extend `Module` so that the list version finds all four
   arrays as well.

    *Adapted from CMU 10-714,
    [Homework 2](https://github.com/dlsyscourse/hw2), Question 2.*

5. [extended] **Depth family.** ● With `MLPConfig` and `build`, construct the
   residual MLP with `num_inputs=1`, `num_outputs=1`, and `num_blocks` set
   to 0, 1, and 4.
    1. Before training, list the layers that `build` stacks when
       `num_blocks=0` and state which functions of $x$ that model can
       represent.
    1. Train each variant with the squared loss, the same optimizer, and
       the same number of epochs on 1,000 points $x$ drawn uniformly from
       $[-3, 3]$ with targets $y = \sin(2x)$. Report a table of parameter
       count and final training loss against `num_blocks`, and explain the
       loss of the `num_blocks=0` variant from the first part.

    *Adapted from CMU 11-785,
    [Homework 1 Part 1](https://deeplearning.cs.cmu.edu/F23/document/homework/HW1/HW1P1_F23.pdf),
    Section 7.*

6. [code] **Lazy input width.** Build
   `net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))`
   and pass a batch with 20 features through it, then a batch with 30
   features. Explain the outcome from what the first forward pass did to
   each layer. Then describe a model that accepts inputs of either width
   while sharing every layer after the first.

[Discussions](https://d2l.discourse.group/t/36752)